# Equities Market Response: Ship Movement + AI Sentiment Analysis

Analyzing how equity markets (US, Europe, Asia) respond to:
- Ship traffic disruptions in Strait of Hormuz
- AI-analyzed geopolitical sentiment
- Regional differences in market response

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import glob
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (20, 12)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# Load shipping data
shipping_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
shipping_df['DateTime'] = pd.to_datetime(shipping_df['DateTime'])
shipping_df.set_index('DateTime', inplace=True)
shipping_df['Total'] = shipping_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)
shipping_df['MA_30'] = shipping_df['Total'].rolling(30).mean()
shipping_df['Disruption_Pct'] = ((shipping_df['Total'] - shipping_df['MA_30']) / shipping_df['MA_30']) * 100

# Load sentiment data
sentiment_df = pd.read_csv('Data/crisis_sentiment_analysis.csv')
sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])

sentiment_daily = pd.read_csv('Data/crisis_sentiment_daily.csv')
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])
sentiment_daily_indexed = sentiment_daily.set_index('date')

# Load equities data by region
equities = {}

# US Equities
us_files = glob.glob('Data/Equities/US/*.parquet')
for file in us_files:
    ticker = file.split('/')[-1].replace('.parquet', '')
    df = pd.read_parquet(file)
    df.index = pd.to_datetime(df.index)
    equities[f'US_{ticker}'] = df['Close']

# Europe Equities
europe_files = glob.glob('Data/Equities/Europe/*.parquet')
for file in europe_files:
    ticker = file.split('/')[-1].replace('.parquet', '')
    df = pd.read_parquet(file)
    df.index = pd.to_datetime(df.index)
    equities[f'EU_{ticker}'] = df['Close']

# Asia Equities
asia_files = glob.glob('Data/Equities/Asia/*.parquet')
for file in asia_files:
    ticker = file.split('/')[-1].replace('.parquet', '')
    df = pd.read_parquet(file)
    df.index = pd.to_datetime(df.index)
    equities[f'ASIA_{ticker}'] = df['Close']

equities_df = pd.DataFrame(equities)

print(f"Loaded {len(equities_df.columns)} equity tickers")
print(f"  US: {len([c for c in equities_df.columns if c.startswith('US_')])}")
print(f"  Europe: {len([c for c in equities_df.columns if c.startswith('EU_')])}")
print(f"  Asia: {len([c for c in equities_df.columns if c.startswith('ASIA_')])}")
print(f"Date range: {equities_df.index.min()} to {equities_df.index.max()}")

## 2. Merge and Prepare Analysis Data

In [ ]:
# Merge all data
analysis_df = equities_df.copy()
analysis_df = analysis_df.join(shipping_df[['Total', 'Disruption_Pct']], how='left')
analysis_df = analysis_df.join(sentiment_daily_indexed[['sentiment_mean', 'article_count']], how='left')

# Forward fill shipping data
analysis_df['Disruption_Pct'] = analysis_df['Disruption_Pct'].fillna(method='ffill')
analysis_df['sentiment_mean'] = analysis_df['sentiment_mean'].fillna(0)

# Calculate returns
for col in equities_df.columns:
    analysis_df[f'{col}_Return'] = analysis_df[col].pct_change()

# Create regional indices
us_cols = [c for c in equities_df.columns if c.startswith('US_')]
eu_cols = [c for c in equities_df.columns if c.startswith('EU_')]
asia_cols = [c for c in equities_df.columns if c.startswith('ASIA_')]

analysis_df['US_Index'] = analysis_df[[f'{c}_Return' for c in us_cols]].mean(axis=1)
analysis_df['EU_Index'] = analysis_df[[f'{c}_Return' for c in eu_cols]].mean(axis=1)
analysis_df['ASIA_Index'] = analysis_df[[f'{c}_Return' for c in asia_cols]].mean(axis=1)

# Crisis indicators
analysis_df['Shipping_Crisis'] = (analysis_df['Disruption_Pct'] < -20).astype(int)
analysis_df['Sentiment_Crisis'] = (analysis_df['sentiment_mean'] < -0.5).astype(int)
analysis_df['Combined_Crisis'] = ((analysis_df['Shipping_Crisis'] == 1) | 
                                   (analysis_df['Sentiment_Crisis'] == 1)).astype(int)

print(f"\nAnalysis DataFrame: {analysis_df.shape}")
print(f"Crisis days: {analysis_df['Combined_Crisis'].sum()}")

## 3. Regional Index Performance

In [ ]:
# Calculate regional performance by scenario
scenarios = {
    'Normal': (analysis_df['Combined_Crisis'] == 0),
    'Shipping Crisis': (analysis_df['Shipping_Crisis'] == 1),
    'Sentiment Crisis': (analysis_df['Sentiment_Crisis'] == 1),
    'Combined Crisis': ((analysis_df['Shipping_Crisis'] == 1) & (analysis_df['Sentiment_Crisis'] == 1))
}

regional_results = {}
for scenario_name, mask in scenarios.items():
    scenario_data = analysis_df[mask]
    regional_results[scenario_name] = {
        'Days': mask.sum(),
        'US': scenario_data['US_Index'].mean() * 252 * 100,
        'Europe': scenario_data['EU_Index'].mean() * 252 * 100,
        'Asia': scenario_data['ASIA_Index'].mean() * 252 * 100,
        'US_Vol': scenario_data['US_Index'].std() * np.sqrt(252) * 100,
        'EU_Vol': scenario_data['EU_Index'].std() * np.sqrt(252) * 100,
        'ASIA_Vol': scenario_data['ASIA_Index'].std() * np.sqrt(252) * 100
    }

regional_df = pd.DataFrame(regional_results).T
print("\nRegional Performance by Scenario:")
print(regional_df.round(2))

In [ ]:
# Visualize regional performance
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# Plot 1: Returns by region and scenario
return_cols = ['US', 'Europe', 'Asia']
regional_df[return_cols].plot(kind='bar', ax=axes[0, 0], width=0.8)
axes[0, 0].set_title('Regional Equity Returns by Scenario', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Annualized Return (%)')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].legend(title='Region')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')

# Plot 2: Volatility by region and scenario
vol_cols = ['US_Vol', 'EU_Vol', 'ASIA_Vol']
regional_df[vol_cols].plot(kind='bar', ax=axes[0, 1], width=0.8)
axes[0, 1].set_title('Regional Equity Volatility by Scenario', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Annualized Volatility (%)')
axes[0, 1].legend(title='Region', labels=['US', 'Europe', 'Asia'])
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha='right')

# Plot 3: Impact (difference from normal)
impact_df = regional_df[return_cols].subtract(regional_df.loc['Normal'][return_cols], axis=1)
impact_df.drop('Normal').plot(kind='bar', ax=axes[1, 0], width=0.8)
axes[1, 0].set_title('Crisis Impact on Regional Returns (vs Normal)', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Return Difference (%)')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].legend(title='Region')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')

# Plot 4: Sharpe ratios
sharpe_df = pd.DataFrame()
for region in ['US', 'Europe', 'Asia']:
    vol_col = f'{region.split()[0] if " " not in region else region}_Vol' if region != 'Europe' else 'EU_Vol'
    sharpe_df[region] = regional_df[region] / regional_df[vol_col]

sharpe_df.plot(kind='bar', ax=axes[1, 1], width=0.8)
axes[1, 1].set_title('Regional Sharpe Ratios by Scenario', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Sharpe Ratio')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 1].legend(title='Region')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 4. Sector Analysis (Energy, Defense, Transportation)

In [ ]:
# Define sector groupings
sectors = {
    'Energy': ['US_XLE', 'US_XOM', 'US_CVX', 'US_COP', 'US_EOG', 'US_HAL', 'US_SLB', 'EU_IEO', 'EU_EQNR'],
    'Defense/Aerospace': ['US_LMT', 'US_BA', 'US_ITA'],
    'Transportation': ['US_JETS', 'US_BDRY'],
    'Shipping/Tankers': ['US_DHT', 'US_FRO', 'US_STNG', 'US_TNK'],
    'LNG': ['US_LNG', 'US_WDS'],
    'Broad Market': ['US_SPY', 'US_XLI', 'EU_EZU', 'ASIA_FXI', 'ASIA_EWJ']
}

sector_results = {}
for sector_name, tickers in sectors.items():
    # Filter tickers that exist in our data
    available_tickers = [t for t in tickers if t in equities_df.columns]
    if not available_tickers:
        continue
    
    return_cols = [f'{t}_Return' for t in available_tickers]
    sector_return = analysis_df[return_cols].mean(axis=1)
    
    sector_results[sector_name] = {}
    for scenario_name, mask in scenarios.items():
        scenario_data = sector_return[mask]
        sector_results[sector_name][scenario_name] = scenario_data.mean() * 252 * 100

sector_df = pd.DataFrame(sector_results).T
print("\nSector Performance by Scenario (Annualized %):")
print(sector_df.round(2))

In [ ]:
# Visualize sector performance
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Plot 1: Sector returns by scenario
sector_df.T.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Sector Performance by Scenario', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Annualized Return (%)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Plot 2: Crisis impact by sector
sector_impact = sector_df.subtract(sector_df['Normal'], axis=0).drop('Normal', axis=1)
sector_impact.T.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title('Sector Crisis Impact (vs Normal)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return Difference (%)')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].legend(title='Sector', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Individual Stock Winners and Losers

In [ ]:
# Calculate performance for all stocks during combined crisis
combined_crisis_mask = (analysis_df['Shipping_Crisis'] == 1) & (analysis_df['Sentiment_Crisis'] == 1)
normal_mask = analysis_df['Combined_Crisis'] == 0

stock_performance = {}
for col in equities_df.columns:
    return_col = f'{col}_Return'
    crisis_ret = analysis_df[combined_crisis_mask][return_col].mean() * 252 * 100
    normal_ret = analysis_df[normal_mask][return_col].mean() * 252 * 100
    stock_performance[col] = {
        'Normal': normal_ret,
        'Crisis': crisis_ret,
        'Impact': crisis_ret - normal_ret
    }

stock_perf_df = pd.DataFrame(stock_performance).T

print("\nTOP 10 WINNERS During Combined Crisis:")
print(stock_perf_df.nlargest(10, 'Crisis')[['Normal', 'Crisis', 'Impact']].round(2))

print("\nTOP 10 LOSERS During Combined Crisis:")
print(stock_perf_df.nsmallest(10, 'Crisis')[['Normal', 'Crisis', 'Impact']].round(2))

## 6. Correlation Analysis

In [ ]:
# Calculate correlations with disruption and sentiment
corr_data = {
    'Shipping_Disruption': [],
    'Sentiment': [],
    'Ticker': []
}

for col in equities_df.columns:
    return_col = f'{col}_Return'
    ship_corr = analysis_df[[return_col, 'Disruption_Pct']].corr().iloc[0, 1]
    sent_corr = analysis_df[[return_col, 'sentiment_mean']].corr().iloc[0, 1]
    
    corr_data['Ticker'].append(col)
    corr_data['Shipping_Disruption'].append(ship_corr)
    corr_data['Sentiment'].append(sent_corr)

corr_df = pd.DataFrame(corr_data).set_index('Ticker')

print("\nMost NEGATIVELY correlated with Shipping Disruption (hurt by crisis):")
print(corr_df.nsmallest(10, 'Shipping_Disruption')['Shipping_Disruption'].round(3))

print("\nMost POSITIVELY correlated with Shipping Disruption (benefit from crisis):")
print(corr_df.nlargest(10, 'Shipping_Disruption')['Shipping_Disruption'].round(3))

## 7. Key Insights

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS - EQUITIES RESPONSE TO SHIPPING + SENTIMENT")
print("="*80)

print("\n1. REGIONAL IMPACT RANKING:")
combined_crisis_regional = regional_df.loc['Combined Crisis'][return_cols].sort_values()
for region, ret in combined_crisis_regional.items():
    impact = ret - regional_df.loc['Normal'][region]
    print(f"   {region}: {ret:+.2f}% (Impact: {impact:+.2f}%)")

print("\n2. BEST PERFORMING SECTORS During Combined Crisis:")
best_sectors = sector_df['Combined Crisis'].sort_values(ascending=False).head(3)
for sector, ret in best_sectors.items():
    print(f"   {sector}: {ret:+.2f}%")

print("\n3. WORST PERFORMING SECTORS During Combined Crisis:")
worst_sectors = sector_df['Combined Crisis'].sort_values().head(3)
for sector, ret in worst_sectors.items():
    print(f"   {sector}: {ret:+.2f}%")

print("\n4. VOLATILITY CHANGES:")
for region in ['US', 'Europe', 'Asia']:
    vol_col = 'US_Vol' if region == 'US' else ('EU_Vol' if region == 'Europe' else 'ASIA_Vol')
    normal_vol = regional_df.loc['Normal'][vol_col]
    crisis_vol = regional_df.loc['Combined Crisis'][vol_col]
    vol_change = ((crisis_vol / normal_vol) - 1) * 100
    print(f"   {region}: {normal_vol:.1f}% → {crisis_vol:.1f}% (+{vol_change:.1f}%)")

print("\n5. TRADING STRATEGY:")
print("   During Shipping + Sentiment Crisis:")
top_winners = stock_perf_df.nlargest(3, 'Impact')
print(f"   - LONG: {', '.join(top_winners.index.tolist())}")
top_losers = stock_perf_df.nsmallest(3, 'Impact')
print(f"   - SHORT: {', '.join(top_losers.index.tolist())}")

print("\n" + "="*80)